In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pptx import Presentation
from pptx.util import Inches, Pt
import warnings

warnings.filterwarnings("ignore")

# Automatically locate project root
current = Path.cwd().resolve()

BASE_DIR = next(
    (p for p in [current, *current.parents]
     if (p / "outputs").exists()),
    None
)

if BASE_DIR is None:
    raise FileNotFoundError(
        "Project root not found. Cannot locate the outputs folder."
    )

OUTPUT_DIR = BASE_DIR / "week-11-day-2-executive-reports"
CHART_DIR = OUTPUT_DIR / "charts"
REPORT_DIR = OUTPUT_DIR / "reports"

CHART_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = BASE_DIR / "outputs" / "cleaned_indian_business_data.csv"
FINDINGS_PATH = BASE_DIR / "outputs" / "business_findings.csv"
OUTLIERS_PATH = BASE_DIR / "outputs" / "outlier_analysis.csv"

print("PROJECT ROOT:", BASE_DIR)
print("DATA PATH:", DATA_PATH)
print("DATA EXISTS:", DATA_PATH.exists())
print("FINDINGS EXISTS:", FINDINGS_PATH.exists())
print("OUTLIERS EXISTS:", OUTLIERS_PATH.exists())

PROJECT ROOT: D:\Cynaris-Internship
DATA PATH: D:\Cynaris-Internship\outputs\cleaned_indian_business_data.csv
DATA EXISTS: True
FINDINGS EXISTS: True
OUTLIERS EXISTS: True


In [4]:
required_files = [DATA_PATH, FINDINGS_PATH, OUTLIERS_PATH]

for file in required_files:
    if not file.exists():
        raise FileNotFoundError(f"Missing required file: {file}")

df = pd.read_csv(DATA_PATH)
findings_df = pd.read_csv(FINDINGS_PATH)
outliers_df = pd.read_csv(OUTLIERS_PATH)

print("Main dataset:", df.shape)
print("Findings dataset:", findings_df.shape)
print("Outlier dataset:", outliers_df.shape)

display(df.head())
display(findings_df.head())
display(outliers_df.head())

Main dataset: (600, 15)
Findings dataset: (8, 6)
Outlier dataset: (8, 2)


,listing_id,city,locality,type,area_sqft,price_per_sqft,total_price_lakh,age_years,floor,parking,furnished,listed_date,days_on_market,transaction_type,sold
0,PROP-0001,Chennai,CHE-Sector-18,4BHK,2507,13243,325.79,12,13,0,Unfurnished,2023-01-18,198,Rent,0
1,PROP-0002,Mumbai,MUM-Sector-5,Commercial,1404,7616,247.88,11,24,1,Fully,2023-07-30,123,Rent,0
2,PROP-0003,Delhi NCR,DEL-Sector-17,Plot,2673,3117,148.91,12,10,0,Fully,2023-05-05,164,Sale,0
3,PROP-0004,Pune,PUN-Sector-28,2BHK,1225,6940,108.03,11,20,0,Fully,2024-09-11,214,Sale,1
4,PROP-0005,Delhi NCR,DEL-Sector-3,Villa,2412,23280,363.66,16,4,1,Unfurnished,2023-05-23,171,Rent,1


,Variable,Mean,Median,Minimum,Maximum,Standard Deviation
0,area_sqft,2204.09,2195.5,406.00,4000.00,1029.90
1,price_per_sqft,10444.07,8401.0,2574.00,23900.00,6214.61
2,total_price_lakh,229.10,172.6,15.71,914.71,183.75
3,age_years,10.02,10.0,0.00,20.00,6.13
4,floor,12.46,13.0,0.00,25.00,7.53


,Column,Outlier_Count
0,area_sqft,0
1,price_per_sqft,0
2,total_price_lakh,28
3,age_years,0
4,floor,0


In [5]:
print("DATASET OVERVIEW")
print("=" * 60)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Duplicate rows:", df.duplicated().sum())
print("Missing values:", df.isnull().sum().sum())

print("\nCOLUMN INFORMATION")
display(pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isnull().sum().values,
    "Unique Values": df.nunique().values
}))

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

print("\nNumeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

DATASET OVERVIEW
Rows: 600
Columns: 15
Duplicate rows: 0
Missing values: 0

COLUMN INFORMATION


,Column,Data Type,Missing Values,Unique Values
0,listing_id,str,0,600
1,city,str,0,8
2,locality,str,0,218
3,type,str,0,7
4,area_sqft,int64,0,546
5,price_per_sqft,int64,0,588
6,total_price_lakh,float64,0,596
7,age_years,int64,0,21
8,floor,int64,0,26
9,parking,int64,0,2



Numeric columns: ['area_sqft', 'price_per_sqft', 'total_price_lakh', 'age_years', 'floor', 'parking', 'days_on_market', 'sold']
Categorical columns: ['listing_id', 'city', 'locality', 'type', 'furnished', 'listed_date', 'transaction_type']


In [6]:
kpis = {
    "Total Records": len(df),
    "Total Columns": len(df.columns),
    "Numeric Columns": len(numeric_cols),
    "Categorical Columns": len(categorical_cols),
    "Duplicate Records": int(df.duplicated().sum()),
    "Missing Cells": int(df.isnull().sum().sum())
}

if numeric_cols:
    first_numeric = numeric_cols[0]
    kpis[f"Average {first_numeric}"] = round(df[first_numeric].mean(), 2)
    kpis[f"Maximum {first_numeric}"] = round(df[first_numeric].max(), 2)
    kpis[f"Minimum {first_numeric}"] = round(df[first_numeric].min(), 2)

kpi_df = pd.DataFrame(
    list(kpis.items()),
    columns=["KPI", "Value"]
)

display(kpi_df)
kpi_df.to_csv(REPORT_DIR / "executive_kpis.csv", index=False)

,KPI,Value
0,Total Records,600.00
1,Total Columns,15.00
2,Numeric Columns,8.00
3,Categorical Columns,7.00
4,Duplicate Records,0.00
5,Missing Cells,0.00
6,Average area_sqft,2204.09
7,Maximum area_sqft,4000.00
8,Minimum area_sqft,406.00


In [7]:
if numeric_cols:
    selected_numeric = numeric_cols[0]

    fig = px.histogram(
        df,
        x=selected_numeric,
        nbins=30,
        title=f"Distribution of {selected_numeric}",
        template="plotly_white",
        marginal="box"
    )

    fig.update_layout(
        xaxis_title=selected_numeric,
        yaxis_title="Frequency",
        title_x=0.5
    )

    fig.show()

    fig.write_html(
        CHART_DIR / "numeric_distribution.html"
    )
else:
    print("No numeric columns available.")

In [8]:
if categorical_cols:
    selected_category = categorical_cols[0]

    category_counts = (
        df[selected_category]
        .fillna("Missing")
        .value_counts()
        .head(10)
        .reset_index()
    )

    category_counts.columns = [selected_category, "Count"]

    fig = px.bar(
        category_counts,
        x=selected_category,
        y="Count",
        title=f"Top 10 Categories: {selected_category}",
        template="plotly_white",
        text="Count"
    )

    fig.update_layout(title_x=0.5)
    fig.show()

    fig.write_html(
        CHART_DIR / "categorical_analysis.html"
    )
else:
    print("No categorical columns available.")

In [9]:
if len(numeric_cols) >= 2:
    correlation = df[numeric_cols].corr()

    fig = px.imshow(
        correlation,
        text_auto=".2f",
        aspect="auto",
        color_continuous_scale="RdBu_r",
        title="Numerical Feature Correlation"
    )

    fig.update_layout(title_x=0.5)
    fig.show()

    fig.write_html(
        CHART_DIR / "correlation_heatmap.html"
    )

    correlation.to_csv(
        REPORT_DIR / "correlation_matrix.csv"
    )
else:
    print("At least two numeric columns are required for correlation analysis.")

In [10]:
print("OUTLIER ANALYSIS")
print("=" * 60)

display(outliers_df.head(20))

if not outliers_df.empty:
    print("Outlier analysis records:", len(outliers_df))

    object_cols = outliers_df.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    if object_cols:
        outlier_category = object_cols[0]

        outlier_counts = (
            outliers_df[outlier_category]
            .fillna("Missing")
            .value_counts()
            .head(10)
            .reset_index()
        )

        outlier_counts.columns = [outlier_category, "Count"]

        fig = px.bar(
            outlier_counts,
            x=outlier_category,
            y="Count",
            title=f"Outlier Analysis: {outlier_category}",
            template="plotly_white"
        )

        fig.show()
        fig.write_html(CHART_DIR / "outlier_analysis.html")

    elif numeric_cols:
        print("Outlier CSV has no categorical column to group by.")
else:
    print("No outlier records found.")

OUTLIER ANALYSIS


,Column,Outlier_Count
0,area_sqft,0
1,price_per_sqft,0
2,total_price_lakh,28
3,age_years,0
4,floor,0
5,parking,0
6,days_on_market,0
7,sold,0


Outlier analysis records: 8


In [11]:
executive_findings = []

executive_findings.append(
    f"The dataset contains {len(df):,} records and {len(df.columns)} columns."
)

executive_findings.append(
    f"The dataset contains {df.isnull().sum().sum():,} missing cells."
)

executive_findings.append(
    f"The dataset contains {df.duplicated().sum():,} duplicate records."
)

if numeric_cols:
    for col in numeric_cols[:3]:
        executive_findings.append(
            f"{col}: mean = {df[col].mean():.2f}, "
            f"minimum = {df[col].min():.2f}, "
            f"maximum = {df[col].max():.2f}."
        )

print("EXECUTIVE FINDINGS")
print("=" * 60)

for i, finding in enumerate(executive_findings, 1):
    print(f"{i}. {finding}")

with open(REPORT_DIR / "executive_findings.txt", "w", encoding="utf-8") as file:
    file.write("\n".join(executive_findings))

EXECUTIVE FINDINGS
1. The dataset contains 600 records and 15 columns.
2. The dataset contains 0 missing cells.
3. The dataset contains 0 duplicate records.
4. area_sqft: mean = 2204.09, minimum = 406.00, maximum = 4000.00.
5. price_per_sqft: mean = 10444.07, minimum = 2574.00, maximum = 23900.00.
6. total_price_lakh: mean = 229.10, minimum = 15.71, maximum = 914.71.


In [12]:
prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)

def add_title(slide, title, subtitle=None):
    title_box = slide.shapes.title
    title_box.text = title

    title_box.text_frame.paragraphs[0].font.size = Pt(28)
    title_box.text_frame.paragraphs[0].font.bold = True

    if subtitle:
        box = slide.shapes.add_textbox(
            Inches(0.7), Inches(1.0),
            Inches(11.5), Inches(0.5)
        )
        box.text_frame.text = subtitle

def add_text(slide, text, left, top, width, height, size=18):
    box = slide.shapes.add_textbox(
        Inches(left), Inches(top),
        Inches(width), Inches(height)
    )

    frame = box.text_frame
    frame.word_wrap = True
    frame.text = str(text)

    for paragraph in frame.paragraphs:
        paragraph.font.size = Pt(size)

# Slide 1: Title
slide = prs.slides.add_slide(prs.slide_layouts[0])
add_title(
    slide,
    "Indian Business Dataset",
    "Executive Analytics Report | Week 11 Day 2"
)

# Slide 2: Executive KPIs
slide = prs.slides.add_slide(prs.slide_layouts[5])
add_title(slide, "Executive KPI Summary")

kpi_text = "\n".join(
    [f"{row.KPI}: {row.Value}" for row in kpi_df.itertuples()]
)

add_text(slide, kpi_text, 0.8, 1.5, 11.5, 5.2, 20)

# Slide 3: Main findings
slide = prs.slides.add_slide(prs.slide_layouts[5])
add_title(slide, "Key Business Findings")

add_text(
    slide,
    "\n\n".join(
        [f"{i}. {finding}" for i, finding in enumerate(executive_findings, 1)]
    ),
    0.8, 1.5, 11.5, 5.2, 17
)

# Slide 4: Existing business findings
slide = prs.slides.add_slide(prs.slide_layouts[5])
add_title(slide, "Business Findings from EDA")

findings_text = findings_df.head(12).to_string(index=False)

add_text(
    slide,
    findings_text,
    0.6, 1.4, 12, 5.5, 12
)

# Slide 5: Outlier analysis
slide = prs.slides.add_slide(prs.slide_layouts[5])
add_title(slide, "Outlier Analysis")

outlier_text = (
    outliers_df.head(15).to_string(index=False)
    if not outliers_df.empty
    else "No outlier records available."
)

add_text(
    slide,
    outlier_text,
    0.6, 1.4, 12, 5.5, 12
)

# Slide 6: Recommendations
slide = prs.slides.add_slide(prs.slide_layouts[5])
add_title(slide, "Recommendations and Next Steps")

recommendations = [
    "Investigate missing values and determine whether they affect key business conclusions.",
    "Review duplicate records before using the dataset for further analysis.",
    "Investigate detected outliers before deciding whether they represent errors or genuine business events.",
    "Use the identified business findings to guide further analysis and decision-making."
]

add_text(
    slide,
    "\n\n".join(
        [f"{i}. {item}" for i, item in enumerate(recommendations, 1)]
    ),
    0.8, 1.5, 11.5, 5.2, 18
)

ppt_path = REPORT_DIR / "W11D2_Indian_Business_Executive_Report.pptx"
prs.save(ppt_path)

print("PowerPoint created:", ppt_path)

PowerPoint created: D:\Cynaris-Internship\week-11-day-2-executive-reports\reports\W11D2_Indian_Business_Executive_Report.pptx


In [13]:
print("FINAL PROJECT VALIDATION")
print("=" * 60)

for folder in [CHART_DIR, REPORT_DIR]:
    print(f"\n{folder.name.upper()}")

    for file in sorted(folder.iterdir()):
        if file.is_file():
            print(f" - {file.name} ({file.stat().st_size:,} bytes)")

assert ppt_path.exists(), "PowerPoint file was not created."
assert ppt_path.stat().st_size > 0, "PowerPoint file is empty."

print("\nW11D2 REPORT GENERATION COMPLETED SUCCESSFULLY.")

FINAL PROJECT VALIDATION

CHARTS
 - categorical_analysis.html (4,827,717 bytes)
 - correlation_heatmap.html (4,828,618 bytes)
 - numeric_distribution.html (4,831,199 bytes)
 - outlier_analysis.html (4,827,515 bytes)

REPORTS
 - correlation_matrix.csv (1,376 bytes)
 - executive_findings.txt (337 bytes)
 - executive_kpis.csv (218 bytes)
 - W11D2_Indian_Business_Executive_Report.pptx (34,188 bytes)

W11D2 REPORT GENERATION COMPLETED SUCCESSFULLY.
